# Guarantee full coverage from the irregular grid

`02_sweep_reduced_fov_path_combinations.ipynb` found all 4 irregular-grid
combinations left real tissue uncovered (418-3687 um^2) on the real
LT066_sample_01/merfish boundary, even with `optimize_offset=False`. This
notebook diagnoses the actual root cause, applies the fix, and validates
that every irregular-grid combination is now coverage-safe.

**Root cause found**: `fix_overlap_clusters` -- REQUIRED post-processing
that redistributes a band's cross-axis positions to remove piece-boundary
overlap clusters -- sized its corrected sub-bands with
`round(span / step_size) + 1`. Rounding DOWN under-provisions FOVs
whenever `span / step_size` sits just under a half-integer, which can
leave a real gap in the corrected sub-band. Fixed in `positions.py` to
`ceil(span / step_size) + 1` (never under-provisions). A second,
independent line of defense, `patch_uncovered_gaps`, was also added and
wired into `build_irregular_boundary_path` -- it measures the true
uncovered area after the full pipeline runs and tiles any real gap that's
still left, regardless of cause.


## 1 — Setup


In [1]:
import os
import sys
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import box as shapely_box
from shapely.ops import unary_union

# notebooks/tests/<subfolder>/ is three levels under the repo root (MERci/).
MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/
SAMPLE_DIR = MERCI_DIR.parent
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.configs   import get_fov_geometry
from MERci.acquisition.positions import (
    load_boundary_polygon, load_hole_polygons,
    build_irregular_bands, fix_overlap_clusters, generate_irregular_scanning_path,
    build_irregular_boundary_path, patch_uncovered_gaps, build_reduced_fov_path,
)
from MERci.visualization import get_merci_figures_dir

NOTEBOOK_NAME = "03_guarantee_irregular_grid_coverage"
DATA_DIR      = MERCI_DIR / "cache" / "tests" / "create_positions" / NOTEBOOK_NAME / "data" / "boundary"
FIGURES_DIR   = get_merci_figures_dir(SAMPLE_DIR, "tests", NOTEBOOK_NAME, subfolder="create_positions")

print(f"DATA_DIR   : {DATA_DIR}")
print(f"FIGURES_DIR: {FIGURES_DIR}")


DATA_DIR   : /n/home06/lsepulvedaduran/251225_LT027_saving_time/MERci/cache/tests/create_positions/03_guarantee_irregular_grid_coverage/data/boundary
FIGURES_DIR: /n/home06/lsepulvedaduran/251225_LT027_saving_time/figures/MERci/tests/create_positions/03_guarantee_irregular_grid_coverage


## 2 — Load the same real boundary as notebook 02

Same real LT066_sample_01/merfish ("lineage_merfish") mosaic-derived
boundary + 11 holes -- copied into this notebook's own `DATA_DIR` (its own
independent local copy, per `NOTEBOOK_GUIDELINES.md`'s portability
convention).


In [2]:
SOURCE_BOUNDARY_DIR = Path(
    "/n/holylfs05/LABS/zhuang_lab/Lab/shared/projects/lineage_tracing/experiments/"
    "LT066_sample_01/merfish/positions/boundaries/from_mosaic"
)
if not DATA_DIR.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for f in sorted(SOURCE_BOUNDARY_DIR.glob("*.txt")):
        shutil.copy2(f, DATA_DIR / f.name)
    print(f"Copied {len(list(DATA_DIR.glob('*.txt')))} boundary/hole file(s).")
else:
    print(f"Using existing local copy: {DATA_DIR}")

boundary_polygon = load_boundary_polygon(DATA_DIR / "boundary_positions.txt")
hole_polygons    = load_hole_polygons(DATA_DIR)
tissue           = boundary_polygon.difference(unary_union(hole_polygons))

pixel_size_um, image_size_px = get_fov_geometry("ST2", "60X")
FOV_SIZE_UM  = pixel_size_um * image_size_px
STEP_SIZE_UM = 182.06208
half         = FOV_SIZE_UM / 2.0

print(f"Tissue area: {tissue.area / 1e6:.2f} mm^2   FOV size: {FOV_SIZE_UM:.2f} um   Step: {STEP_SIZE_UM:.2f} um")


Copied 16 boundary/hole file(s).
Tissue area: 30.60 mm^2   FOV size: 202.29 um   Step: 182.06 um


## 3 — Diagnosis: where does the gap actually come from?

Tests the pipeline one stage at a time (both `fixed_axis` values) to
isolate exactly which stage introduces the gap: coverage is measured as
`tissue.difference(union of every FOV's own square)`.


In [3]:
def coverage_after(fixed_axis, apply_fix, fix_fn=fix_overlap_clusters):
    bands = build_irregular_bands(boundary_polygon, hole_polygons, STEP_SIZE_UM, FOV_SIZE_UM,
                                   fixed_axis=fixed_axis)
    if apply_fix:
        bands = [(fv, fix_fn(cv, FOV_SIZE_UM, STEP_SIZE_UM)[0]) for fv, cv in bands]
    path = generate_irregular_scanning_path(bands, fixed_axis=fixed_axis)
    boxes = [shapely_box(x - half, y - half, x + half, y + half) for x, y in path]
    uncovered = tissue.difference(unary_union(boxes))
    return len(path), uncovered.area

def _fix_overlap_clusters_old_buggy(cross_vals, fov_size_um, step_size, bad_overlap_frac=0.3):
    # Historical reproduction of the ORIGINAL (buggy) redistribution, kept
    # here ONLY to demonstrate the bug this notebook fixes -- positions.py
    # itself no longer has this version (see its git history). The single
    # difference from the current, fixed positions.py is `round()` here vs.
    # `ceil()` there.
    cross_vals = np.array(sorted(cross_vals), dtype=float)
    n = len(cross_vals)
    if n < 2:
        return cross_vals.copy(), []
    gaps = np.diff(cross_vals)
    overlap_frac = np.clip((fov_size_um - gaps) / fov_size_um, 0.0, None)
    disjoint = overlap_frac <= 0.0
    is_bad = overlap_frac > bad_overlap_frac
    sub_ranges, start = [], 0
    for i, d in enumerate(disjoint):
        if d:
            sub_ranges.append((start, i)); start = i + 1
    sub_ranges.append((start, n - 1))
    fixed = []
    for lo, hi in sub_ranges:
        seg_bad = is_bad[lo:hi] if hi > lo else np.array([], dtype=bool)
        if hi == lo or not seg_bad.any():
            fixed.extend(cross_vals[lo:hi + 1].tolist()); continue
        span = cross_vals[hi] - cross_vals[lo]
        n_before = hi - lo + 1
        expected_n = max(2, int(round(span / step_size)) + 1)   # <-- the bug: round(), not ceil()
        n_after = min(n_before, expected_n)
        fixed.extend(np.linspace(cross_vals[lo], cross_vals[hi], n_after).tolist())
    return np.array(sorted(fixed)), []

rows = []
for fixed_axis in ("y", "x"):
    n0, u0 = coverage_after(fixed_axis, apply_fix=False)
    n1, u1 = coverage_after(fixed_axis, apply_fix=True, fix_fn=_fix_overlap_clusters_old_buggy)
    n2, u2 = coverage_after(fixed_axis, apply_fix=True, fix_fn=fix_overlap_clusters)
    rows += [
        (fixed_axis, "before fix_overlap_clusters",              n0, u0),
        (fixed_axis, "after OLD buggy round()-based fix",         n1, u1),
        (fixed_axis, "after CURRENT ceil()-based fix (positions.py)", n2, u2),
    ]
    print(f"fixed_axis={fixed_axis}")
    print(f"  before fix_overlap_clusters                  : n={n0:4d}  uncovered={u0:8.1f} um^2")
    print(f"  after OLD buggy round()-based fix             : n={n1:4d}  uncovered={u1:8.1f} um^2")
    print(f"  after CURRENT ceil()-based fix (positions.py) : n={n2:4d}  uncovered={u2:8.1f} um^2")


fixed_axis=y
  before fix_overlap_clusters                  : n=1153  uncovered=     0.0 um^2
  after OLD buggy round()-based fix             : n=1118  uncovered=  3186.3 um^2
  after CURRENT ceil()-based fix (positions.py) : n=1130  uncovered=     0.0 um^2


fixed_axis=x
  before fix_overlap_clusters                  : n=1161  uncovered=     0.0 um^2
  after OLD buggy round()-based fix             : n=1129  uncovered=  2372.3 um^2
  after CURRENT ceil()-based fix (positions.py) : n=1141  uncovered=     0.0 um^2


**Conclusion**: coverage is already perfect right out of
`build_irregular_bands`/`generate_irregular_scanning_path` -- the gap is
introduced entirely by `fix_overlap_clusters`'s old `round()`-based
`expected_n`. Switching to `ceil()` (now in `positions.py`) closes it
completely, with only a few more FOVs than the buggy version (which was
under-counting, not legitimately saving FOVs).


## 4 — Confirm `patch_uncovered_gaps` is now a no-op (defense-in-depth)

With the root cause fixed, the safety-net patch step
(`build_irregular_boundary_path`'s `guarantee_coverage=True` default)
should add nothing on this boundary -- confirms it's a true second line of
defense, not doing the actual work.


In [4]:
for fixed_axis in ("y", "x"):
    path = build_irregular_boundary_path(boundary_polygon, hole_polygons, STEP_SIZE_UM, FOV_SIZE_UM,
                                          fixed_axis=fixed_axis, return_side=None)
    boxes = [shapely_box(x - half, y - half, x + half, y + half) for x, y in path]
    uncovered = tissue.difference(unary_union(boxes)).area
    patched, n_added = patch_uncovered_gaps(path, tissue, STEP_SIZE_UM, FOV_SIZE_UM)
    print(f"fixed_axis={fixed_axis}: build_irregular_boundary_path -> {len(path)} FOVs, "
          f"{uncovered:.4f} um^2 uncovered, patch_uncovered_gaps adds {n_added} more "
          f"(0 expected -- ceil() fix alone already closes the gap here)")


fixed_axis=y: build_irregular_boundary_path -> 1130 FOVs, 0.0000 um^2 uncovered, patch_uncovered_gaps adds 0 more (0 expected -- ceil() fix alone already closes the gap here)


fixed_axis=x: build_irregular_boundary_path -> 1141 FOVs, 0.0000 um^2 uncovered, patch_uncovered_gaps adds 0 more (0 expected -- ceil() fix alone already closes the gap here)


## 5 — Re-validate the full 6-combination sweep against the fixed code

Same sweep as `02_sweep_reduced_fov_path_combinations.ipynb`'s own section
4, re-run here against the fixed `positions.py` for a direct, self-
contained before/after record in this notebook.


In [5]:
COMBOS = [
    dict(label="regular",                          irregular_grid=False, optimize_offset=False, fixed_axis=None),
    dict(label="regular + optimize_offset",         irregular_grid=False, optimize_offset=True,  fixed_axis=None),
    dict(label="irregular (fixed_axis=y)",          irregular_grid=True,  optimize_offset=False, fixed_axis="y"),
    dict(label="irregular (fixed_axis=y) + offset", irregular_grid=True,  optimize_offset=True,  fixed_axis="y"),
    dict(label="irregular (fixed_axis=x)",          irregular_grid=True,  optimize_offset=False, fixed_axis="x"),
    dict(label="irregular (fixed_axis=x) + offset", irregular_grid=True,  optimize_offset=True,  fixed_axis="x"),
]
COVERAGE_EPS_UM2 = FOV_SIZE_UM ** 2 / 1000

results = {}
for combo in COMBOS:
    kwargs = dict(step_size=STEP_SIZE_UM, fov_size_um=FOV_SIZE_UM,
                  irregular_grid=combo["irregular_grid"], optimize_offset=combo["optimize_offset"])
    if combo["fixed_axis"] is not None:
        kwargs["fixed_axis"] = combo["fixed_axis"]
    res = build_reduced_fov_path(boundary_polygon, hole_polygons, **kwargs)
    results[combo["label"]] = res
    safe = res.redundant.uncovered_area_um2 < COVERAGE_EPS_UM2
    print(f"{combo['label']:32s} n_fovs_final: {len(res.coords):4d}   "
          f"uncovered: {res.redundant.uncovered_area_um2:8.4f} um^2   coverage_safe: {safe}")

all_safe = all(r.redundant.uncovered_area_um2 < COVERAGE_EPS_UM2 for r in results.values())
print(f"\nAll 6 combinations coverage-safe: {all_safe}")
winner = min(results.items(), key=lambda kv: len(kv[1].coords))
print(f"Lowest FOV count overall: '{winner[0]}' -> {len(winner[1].coords)} FOVs")


regular                          n_fovs_final: 1096   uncovered:   1.3340 um^2   coverage_safe: True


regular + optimize_offset        n_fovs_final: 1099   uncovered:   0.0000 um^2   coverage_safe: True


irregular (fixed_axis=y)         n_fovs_final: 1099   uncovered:   0.0000 um^2   coverage_safe: True


irregular (fixed_axis=y) + offset n_fovs_final: 1105   uncovered:   0.0000 um^2   coverage_safe: True


irregular (fixed_axis=x)         n_fovs_final: 1113   uncovered:   0.0175 um^2   coverage_safe: True


irregular (fixed_axis=x) + offset n_fovs_final: 1104   uncovered:   1.7948 um^2   coverage_safe: True

All 6 combinations coverage-safe: True
Lowest FOV count overall: 'regular' -> 1096 FOVs


## 6 — Takeaways

- **Root cause**: `fix_overlap_clusters` (required post-processing for
  every irregular-grid path) sized its corrected sub-bands with
  `round(span / step_size) + 1` -- rounding down under-provisions FOVs
  whenever `span / step_size` sits just under a half-integer, leaving a
  real gap. Verified directly: coverage is perfect immediately after
  `build_irregular_bands`/`generate_irregular_scanning_path` (before
  `fix_overlap_clusters` runs at all); the gap appears only after the old
  `round()`-based redistribution.

- **Fix**: `positions.py`'s `fix_overlap_clusters` now uses
  `ceil(span / step_size) + 1`, which by construction never leaves a gap
  wider than `step_size` inside a corrected sub-band. Confirmed: 0.0 um^2
  uncovered on this real boundary for both `fixed_axis` values, at the
  cost of only a handful more FOVs than the buggy version reported (which
  was under-counting, not genuinely saving FOVs).

- **`patch_uncovered_gaps` (new function, wired into
  `build_irregular_boundary_path` by default)**: a second, independent
  line of defense that measures the real uncovered area after the whole
  pipeline runs and tiles any gap that's still left, regardless of cause.
  Confirmed a true no-op on this boundary now that the root cause is
  fixed (adds 0 FOVs) -- kept anyway so "guarantee full coverage" doesn't
  depend on trusting the upstream arithmetic alone.

- **`02_sweep_reduced_fov_path_combinations.ipynb`, re-run against the
  fix**: all 6 parameter combinations are now coverage-safe. Ranking is
  unchanged from before the bug was found -- `regular` (no offset
  optimization) still has the lowest FOV count -- but the comparison is
  now actually fair: no combination wins by skipping real tissue.
